In [7]:
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import json


# Paths

PROJECT_ROOT = Path.cwd().parent
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "sample_features.csv"

FORECAST_PATH = PROJECT_ROOT / "models" / "artifacts" / "xgb_forecast_final.csv"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics" / "xgb_metrics_final.json"

FORECAST_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Load & clean data

df = pd.read_csv(FEATURES_PATH, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)


# Gérer les doublons

if df['date'].duplicated().sum() > 0:
    print(f"[INFO] {df['date'].duplicated().sum()} doublons détectés, moyenne par date")
    df = df.groupby('date').mean().reset_index()


df = df.set_index('date').asfreq('D')
df = df.interpolate()  
assert df.index.is_unique, "Erreur : index toujours non-unique après nettoyage !"

horizon = 30
lags = [1, 7, 14]
roll_windows = [7, 14]

for lag in lags:
    df[f'y_lag_{lag}'] = df['value'].shift(lag)

for window in roll_windows:
    df[f'y_rollmean_{window}'] = df['value'].shift(1).rolling(window).mean()

df_clean = df.dropna()
y = df_clean['value']
X = df_clean.drop(columns=['value'])

print(f"[INFO] Shape après création features : X={X.shape}, y={y.shape}")


# Train/Test split

X_train, X_test = X.iloc[:-horizon], X.iloc[-horizon:]
y_train, y_test = y.iloc[:-horizon], y.iloc[-horizon:]
dates_test = X.iloc[-horizon:].index


# Train XGBoost

model = XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# Predict

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)


# Metrics train/test

metrics = {
    "MAE_train": float(mean_absolute_error(y_train, y_pred_train)),
    "RMSE_train": float(np.sqrt(mean_squared_error(y_train, y_pred_train))),
    "MAE_test": float(mean_absolute_error(y_test, y_pred_test)),
    "RMSE_test": float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
    "num_train_samples": len(y_train),
    "num_test_samples": len(y_test)
}

print("Metrics train/test XGBoost :", metrics)

# Cross-validation TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
cv_mae = []

for train_idx, test_idx in tscv.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[test_idx]
    model_cv = XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    )
    model_cv.fit(X_tr, y_tr)
    y_pred_cv = model_cv.predict(X_val)
    mae_fold = np.mean(np.abs(y_val - y_pred_cv))
    cv_mae.append(mae_fold)

metrics['CV_mae_mean'] = float(np.mean(cv_mae))
metrics['CV_mae_folds'] = cv_mae
print("\n Metrics CV TimeSeriesSplit :", metrics['CV_mae_mean'])



forecast_train = X_train.copy()
forecast_train['yhat'] = y_pred_train
forecast_train['value'] = y_train.values

forecast_test = X_test.copy()
forecast_test['yhat'] = y_pred_test
forecast_test['value'] = y_test.values

forecast = pd.concat([forecast_train, forecast_test], axis=0)
forecast.reset_index(inplace=True)
forecast.rename(columns={'index':'date'}, inplace=True)

forecast.to_csv(FORECAST_PATH, index=False)
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Forecast sauvegardé : {FORECAST_PATH}")
print(f"Metrics sauvegardées : {METRICS_PATH}")


[INFO] 4008 doublons détectés, moyenne par date
[INFO] Shape après création features : X=(351, 22), y=(351,)
Metrics train/test XGBoost : {'MAE_train': 0.0026274187463792816, 'RMSE_train': 0.00375607847805006, 'MAE_test': 0.14672056833902994, 'RMSE_test': 0.274966166322568, 'num_train_samples': 321, 'num_test_samples': 30}

 Metrics CV TimeSeriesSplit : 0.224626139737471
Forecast sauvegardé : c:\Users\nzime\OneDrive\Documents\Prediction_de_vente\models\artifacts\xgb_forecast_final.csv
Metrics sauvegardées : c:\Users\nzime\OneDrive\Documents\Prediction_de_vente\models\metrics\xgb_metrics_final.json
